In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [2]:
# Lecture 3 — Class Exercise: CO2 Emissions

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

print("Libraries imported successfully!")

Libraries imported successfully!


In [3]:
# Download CO2 data from Our World in Data

url = "https://raw.githubusercontent.com/owid/co2-data/master/owid-co2-data.csv"

owid = pd.read_csv(url)

print("Dataset loaded successfully!")
print("Rows:", len(owid))
print("Columns:", owid.columns.tolist())

Dataset loaded successfully!
Rows: 50411
Columns: ['country', 'year', 'iso_code', 'population', 'gdp', 'cement_co2', 'cement_co2_per_capita', 'co2', 'co2_growth_abs', 'co2_growth_prct', 'co2_including_luc', 'co2_including_luc_growth_abs', 'co2_including_luc_growth_prct', 'co2_including_luc_per_capita', 'co2_including_luc_per_gdp', 'co2_including_luc_per_unit_energy', 'co2_per_capita', 'co2_per_gdp', 'co2_per_unit_energy', 'coal_co2', 'coal_co2_per_capita', 'consumption_co2', 'consumption_co2_per_capita', 'consumption_co2_per_gdp', 'cumulative_cement_co2', 'cumulative_co2', 'cumulative_co2_including_luc', 'cumulative_coal_co2', 'cumulative_flaring_co2', 'cumulative_gas_co2', 'cumulative_luc_co2', 'cumulative_oil_co2', 'cumulative_other_co2', 'energy_per_capita', 'energy_per_gdp', 'flaring_co2', 'flaring_co2_per_capita', 'gas_co2', 'gas_co2_per_capita', 'ghg_excluding_lucf_per_capita', 'ghg_per_capita', 'land_use_change_co2', 'land_use_change_co2_per_capita', 'methane', 'methane_per_capi

In [4]:
course_countries = [
    "United States",
    "China",
    "India",
    "Germany",
    "United Kingdom",
    "France",
    "Brazil",
    "Japan",
    "Canada",
    "Australia",
    "South Korea",
    "Russia",
    "South Africa",
    "Mexico",
    "Indonesia"
]

# Select the required years and country records
df = owid[
    (owid["year"] >= 2000) &
    (owid["year"] <= 2022) &
    (owid["country"].isin(course_countries))
].copy()

# Rename columns to match the exercise
df = df.rename(columns={
    "country": "Country",
    "year": "Year",
    "co2": "CO2_Mt",
    "co2_per_capita": "CO2_per_capita"
})

# Assign the regions used in the course dataset
region_map = {
    "United States": "North America",
    "Canada": "North America",
    "China": "Asia",
    "India": "Asia",
    "Japan": "Asia",
    "South Korea": "Asia",
    "Indonesia": "Asia",
    "Germany": "Europe",
    "United Kingdom": "Europe",
    "France": "Europe",
    "Russia": "Europe",
    "Brazil": "Latin America",
    "Mexico": "Latin America",
    "South Africa": "Africa",
    "Australia": "Oceania"
}

df["Region"] = df["Country"].map(region_map)

# Remove records without the measurements needed for the charts
df["CO2_Mt"] = pd.to_numeric(df["CO2_Mt"], errors="coerce")
df["CO2_per_capita"] = pd.to_numeric(
    df["CO2_per_capita"], errors="coerce"
)

print("Countries found:", df["Country"].nunique())
print("Year range:", df["Year"].min(), "-", df["Year"].max())
print("Rows:", len(df))

display(df.head())


Countries found: 15
Year range: 2000 - 2022
Rows: 345


,Country,Year,iso_code,population,gdp,cement_co2,cement_co2_per_capita,CO2_Mt,co2_growth_abs,co2_growth_prct,...,share_of_temperature_change_from_ghg,temperature_change_from_ch4,temperature_change_from_co2,temperature_change_from_ghg,temperature_change_from_n2o,total_ghg,total_ghg_excluding_lucf,trade_co2,trade_co2_share,Region
3654,Australia,2000,AUS,19132472.0,6.920593e+11,3.621,0.189,349.898,6.021,1.751,...,1.660,0.007,0.010,0.019,0.002,771.373,415.179,-57.841,-16.531,Oceania
3655,Australia,2001,AUS,19365438.0,7.138901e+11,3.541,0.183,357.668,7.770,2.221,...,1.667,0.007,0.010,0.020,0.002,736.087,427.062,-56.897,-15.908,Oceania
3656,Australia,2002,AUS,19595356.0,7.470657e+11,3.488,0.178,362.425,4.757,1.330,...,1.669,0.007,0.010,0.020,0.002,700.182,432.796,-41.483,-11.446,Oceania
3657,Australia,2003,AUS,19819449.0,7.744510e+11,3.584,0.181,370.784,8.359,2.307,...,1.666,0.008,0.010,0.020,0.002,734.551,439.491,-28.280,-7.627,Oceania
3658,Australia,2004,AUS,20046679.0,8.097924e+11,3.555,0.177,382.005,11.220,3.026,...,1.667,0.008,0.011,0.020,0.002,761.870,450.416,-30.294,-7.930,Oceania


In [5]:
# TASK 1 — CO2 per capita for all countries
# Highlight China and show other countries in grey

highlight_country = "China"

# Keep rows with available per-capita measurements
task1_df = df.dropna(subset=["CO2_per_capita"]).copy()
task1_df = task1_df.sort_values("Year")

fig1 = go.Figure()

# Draw all other countries in light grey
for country in sorted(task1_df["Country"].unique()):

    if country == highlight_country:
        continue

    country_data = task1_df[
        task1_df["Country"] == country
    ].sort_values("Year")

    fig1.add_trace(go.Scatter(
        x=country_data["Year"],
        y=country_data["CO2_per_capita"],
        mode="lines",
        line=dict(color="#EEEEEE", width=1.5),
        showlegend=False,
        hovertemplate=(
            country
            + "<br>Year: %{x}"
            + "<br>CO2 per capita: %{y:.2f} tonnes"
            + "<extra></extra>"
        )
    ))

# Draw China in red
china1 = task1_df[
    task1_df["Country"] == highlight_country
].sort_values("Year")

if not china1.empty:

    fig1.add_trace(go.Scatter(
        x=china1["Year"],
        y=china1["CO2_per_capita"],
        mode="lines",
        line=dict(color="#D62728", width=3),
        showlegend=False,
        hovertemplate=(
            "China"
            + "<br>Year: %{x}"
            + "<br>CO2 per capita: %{y:.2f} tonnes"
            + "<extra></extra>"
        )
    ))

    # Label China at the end of its line
    fig1.add_annotation(
        x=china1["Year"].iloc[-1],
        y=china1["CO2_per_capita"].iloc[-1],
        text="<b>China</b>",
        showarrow=False,
        xanchor="left",
        xshift=8,
        font=dict(color="#D62728", size=13)
    )

# Clean chart design
fig1.update_layout(
    title=dict(
        text="China's CO2 per Capita Has Changed Over Time",
        x=0.02,
        xanchor="left"
    ),
    template="plotly_white",
    showlegend=False,
    xaxis=dict(
        title="Year",
        range=[2000, 2025],
        dtick=5,
        showgrid=False,
        showline=False,
        zeroline=False
    ),
    yaxis=dict(
        title="CO2 emissions per person (tonnes)",
        showgrid=True,
        gridcolor="#F2F2F2",
        showline=False,
        zeroline=False
    ),
    margin=dict(l=75, r=100, t=90, b=60),
    hovermode="x unified"
)

fig1.show()

In [6]:

# Select Asian countries
asia_df = df[df["Region"] == "Asia"].copy()

print("Asian countries:")
print(asia_df["Country"].unique())

display(asia_df.head())


Asian countries:
['China' 'India' 'Indonesia' 'Japan' 'South Korea']


,Country,Year,iso_code,population,gdp,cement_co2,cement_co2_per_capita,CO2_Mt,co2_growth_abs,co2_growth_prct,...,share_of_temperature_change_from_ghg,temperature_change_from_ch4,temperature_change_from_co2,temperature_change_from_ghg,temperature_change_from_n2o,total_ghg,total_ghg_excluding_lucf,trade_co2,trade_co2_share,Region
9909,China,2000,CHN,1.269581e+09,5.952682e+12,240.210,0.189,3643.810,93.861,2.644,...,8.360,0.030,0.061,0.096,0.005,5282.098,4281.563,-407.806,-11.192,Asia
9910,China,2001,CHN,1.278725e+09,6.329737e+12,266.198,0.208,3724.114,80.305,2.204,...,8.453,0.031,0.063,0.099,0.005,5282.242,4380.003,-392.931,-10.551,Asia
9911,China,2002,CHN,1.286867e+09,6.814494e+12,277.856,0.216,4098.181,374.066,10.044,...,8.551,0.032,0.065,0.102,0.005,5600.191,4772.301,-478.475,-11.675,Asia
9912,China,2003,CHN,1.294517e+09,7.248623e+12,347.633,0.269,4835.251,737.071,17.985,...,8.668,0.032,0.067,0.105,0.006,6373.199,5576.535,-815.563,-16.867,Asia
9913,China,2004,CHN,1.302100e+09,7.830953e+12,362.099,0.278,5210.961,375.710,7.770,...,8.801,0.033,0.069,0.108,0.006,6840.591,6015.262,-956.318,-18.352,Asia


In [7]:
# TASK 2 — CO2 emissions for Asian countries
# Highlight China and show all other Asian countries in grey

highlight_country = "China"

task2_df = asia_df.dropna(subset=["CO2_Mt"]).copy()
task2_df = task2_df.sort_values("Year")

fig2 = go.Figure()

# Draw all other Asian countries in light grey
for country in sorted(task2_df["Country"].unique()):

    if country == highlight_country:
        continue

    country_data = task2_df[
        task2_df["Country"] == country
    ].sort_values("Year")

    fig2.add_trace(go.Scatter(
        x=country_data["Year"],
        y=country_data["CO2_Mt"],
        mode="lines",
        line=dict(color="#EEEEEE", width=1.5),
        showlegend=False,
        hovertemplate=(
            country
            + "<br>Year: %{x}"
            + "<br>CO2 emissions: %{y:,.1f} Mt"
            + "<extra></extra>"
        )
    ))

# Draw China in red
china2 = task2_df[
    task2_df["Country"] == highlight_country
].sort_values("Year")

if not china2.empty:

    fig2.add_trace(go.Scatter(
        x=china2["Year"],
        y=china2["CO2_Mt"],
        mode="lines",
        line=dict(color="#D62728", width=3),
        showlegend=False,
        hovertemplate=(
            "China"
            + "<br>Year: %{x}"
            + "<br>CO2 emissions: %{y:,.1f} Mt"
            + "<extra></extra>"
        )
    ))

    # Label China directly at the end of the line
    fig2.add_annotation(
        x=china2["Year"].iloc[-1],
        y=china2["CO2_Mt"].iloc[-1],
        text="<b>China</b>",
        showarrow=False,
        xanchor="left",
        xshift=8,
        font=dict(color="#D62728", size=13)
    )

# Clean chart design
fig2.update_layout(
    title=dict(
        text="China's CO2 Emissions Rose Sharply, 2000–2022",
        x=0.02,
        xanchor="left"
    ),
    template="plotly_white",
    showlegend=False,
    xaxis=dict(
        title="Year",
        range=[2000, 2025],
        dtick=5,
        showgrid=False,
        showline=False,
        zeroline=False
    ),
    yaxis=dict(
        title="CO2 emissions (million tonnes)",
        showgrid=True,
        gridcolor="#F2F2F2",
        showline=False,
        zeroline=False
    ),
    margin=dict(l=75, r=100, t=90, b=60),
    hovermode="x unified"
)

fig2.show()